# 8.3 중간고사 대비 총정리 — 복습 자가진단 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter08_3_review_diagnostic.ipynb)

책 본문: [8.3 중간고사 대비 총정리](https://smhanlab.com/book-ml/kor/ml1/chapter08/3.html)

본문이 "수식으로 답할 수 있는가"라고 세 번 묻는, 시험에서 가장 자주 나오는
유도 세 종류(로지스틱회귀 1스텝, SVM의 KKT 조건, 두 항 구조)를 **코드로
검증**하는 노트북입니다. 본문 §"손으로 한 번"에서 손으로 계산한 숫자가 여기
서도 똑같이 나오는지 확인하고, Ch04·06의 편향-분산 스펙트럼을 diabetes
데이터(442건)로 한눈에 보고, Ch03의 GDA↔로지스틱회귀 동치성을 실제
데이터에서 시험해 봅니다. numpy/scikit-learn/matplotlib만 쓰며 모든 시드가
고정되어 있습니다. (Colab에서는 첫 셀의 `IMG` 경로를 `/tmp`로 바꾸면 됩니다.)

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)

## 1. 유도 템플릿 ① 검증: 로지스틱회귀 경사하강 1스텝

본문 §"손으로 한 번: 유도 템플릿 ①"의 3샘플 예 \\(x=(1,2,3),
y=(0,1,1), w=0.1, b=0, \\eta=0.5\\)를 그대로 계산한다. \\(z=wx+b\\)가
\\((0.1, 0.2, 0.3)\\)이므로 시그모이드 값은 \\(\\hat p =
(0.5250, 0.5498, 0.5744)\\)여야 한다. 손실은 **샘플당 평균**
\\(\\ell = \\tfrac13\\sum_i[-y_i\\log\\hat p_i-(1-y_i)\\log(1-\\hat p_i)]\\)
이므로 경사는 \\(\\nabla_\\ell(w)=\\tfrac13\\sum_i (\\hat p^{(i)}-y^{(i)})
x^{(i)}\\) — 여기에서 대입하면 \\(-0.5507\\)가 나와야 한다. **경사의
부호**가 핵심이다 — 샘플 1(실제 0인데 0.525로 예측)은
\\(+0.525\\cdot 1\\)의 양수 기여를 해서 \\(w\\)를 **작아지는**
쪽으로 당긴다.

In [2]:
x = np.array([1.0, 2.0, 3.0]); y = np.array([0.0, 1.0, 1.0])
w, b, eta = 0.1, 0.0, 0.5
sig = lambda z: 1.0 / (1.0 + np.exp(-z))

p = sig(w * x + b)
print("σ(z_i):", np.round(p, 4), " <- 본문과 같은가? (0.5250, 0.5498, 0.5744)")
grad_w = np.mean((p - y) * x)     # 유도 템플릿 ①의 마지막 줄 (평균 손실이므로 /n)
grad_b = np.mean(p - y)
w_new = w - eta * grad_w
b_new = b - eta * grad_b
print("∇_w = %.4f   ∇_b = %.4f" % (grad_w, grad_b))
print("w: 0.1 -> %.4f    b: 0.0 -> %.4f" % (w_new, b_new))

σ(z_i): [0.525  0.5498 0.5744]  <- 본문과 같은가? (0.5250, 0.5498, 0.5744)
∇_w = -0.5507   ∇_b = -0.1169
w: 0.1 -> 0.3753    b: 0.0 -> 0.0585


**"미분 없이" 대입한 경사가 정말 미분인 것**인지 유한차분(Ch02의
경사하강이 동작하는 근거)으로 교차 검증한다. 손실 \(\ell(w)=
\sum_i[-y_i\log\hat p_i-(1-y_i)\log(1-\hat p_i)]\)를 \(w\)의 함수로
직접 짜서, \((\ell(w+\delta)-\ell(w-\delta))/2\delta\)가 분석적
경사 \(-0.5507\)와 같은지 확인한다. 시험에서 "이 경사가 왜 맞는지"를
묻는다면 이 유한차분 논리가 답의 뼈대다 — 미분의 정의가
"미소 변화에 대한 손실의 변화율"이기 때문이다.

In [3]:
def logistic_loss(w, b, x, y):   # 평균 손실 (본문과 같은 표기)
    p = sig(w * x + b)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return np.mean(-y * np.log(p) - (1 - y) * np.log(1 - p))

d = 1e-5
numerical = (logistic_loss(w + d, b, x, y) - logistic_loss(w - d, b, x, y)) / (2 * d)
print("분석적 경사   = %.6f" % grad_w)
print("유한차분 경사 = %.6f" % numerical)
print("일치 = %s  (차이 %.2e)" % (abs(grad_w - numerical) < 1e-4, abs(grad_w - numerical)))

분석적 경사   = -0.550675
유한차분 경사 = -0.550675
일치 = True  (차이 1.34e-11)


## 2. 유도 템플릿 ② 검증: SVM의 KKT 조건이 실제로 성립하는가

본문 §"손으로 한 번: 유도 템플릿 ②"의 1차원 장난감 문제 — \(+1\)클래스가
\(-1\)의 두 점(A,B), \(-1\)클래스가 \(+2\)(C)와 **이상치** \(-0.2\)(E) —
을 KKT 세 조건으로 \(C\)의 함수로 손으로 풀면 해는 4개 구간으로 나뉜다:

- **구간 I** (\(C\le 10/57\approx0.175\)): αᵢ가 전부 \(C\),
  \(\|w\|^2=14.44\,C^2\). 예산이 작아 모든 점이 상한에 꽉 찬다 —
  "C를 올리면 마진이 커진다"는 직감이 유일하게 맞는 구간.
- **구간 II** (\(10/57<C<5/6\)): 이상치 E의 위반이 \(\alpha_E=C\)로 최대
  배정되고 경계가 \(w=-\tfrac23,\ b=\tfrac13\)로 **정지** —
  \(\|w\|^2=4/9\), 마진 3.0(=E를 무시하고 +1군(−1)과 C(+2) 사이) 고정.
- **구간 III** (\(5/6<C<25/8\)): \(C(+2)\)가 마진에서 빠지고
  (\(\alpha_C=0\)), 경계가 \(w=-0.8C\)로 **왼쪽으로 미끄러우며** E를
  감싼다(\(0.64C^2\)). 마진은 \(2.5/C\)로 **줄어든다**. \(C=25/16\)에서
  E가 마침내 올바르게 분류된다.
- **구간 IV** (\(C\ge 25/8=3.125\)): E도 마진 위에 오면 해가
  \(w=-\tfrac52,\ b=-\tfrac32\)로 **다시 정지** — \(\|w\|^2=6.25\),
  마진 0.8(=A(−1)과 이상치 E(−0.2) 사이), 경계 x=−0.6. **C를 100으로
  올려도 변하지 않는다.**

핵심: **C가 바꾸는 것은 마진의 *크기*가 아니라 "이상치 E의 위반을
얼마까지 사들이는가"**다. 달성 가능한 마진의 범위(최대 3.0, 이상치를
감내하면 0.8)는 지오메트리가 정하고, C는 그 두 극단 사이의 트레이드오프를
골 뿐이다. "C를 키우면 마진이 무한히 커진다"는 직감은 구간 IV에서
꺾인다.

sklearn의 `SVC`가 이 4구간을 실제로 만드는지 \(C\) 스위프로 확인한다.
(sklearn은 \(\sum\alpha_i=0\) 제약 때문에 `dual_coef_`가
\(\alpha_i y_i\)의 부호를 갖는다 — \(\alpha_A\)(+1클래스)는 +로,
\(\alpha_E\)(−1클래스)는 −로 나타난다. `dual_coef_ @ X_support`가
정확히 \(w\)를 재구성하는 것을 이용해 αᵢ를 읽어낸다. 본문의 αᵢ와
"어떤 점이 SV인가" 결과는 같다.)

In [4]:
from sklearn.svm import SVC

X = np.array([[-1.0, 1.0], [-1.0, -1.0], [2.0, 0.0], [-0.2, 0.0]])
y = np.array([1, 1, -1, -1])

print("C        w1        b        nSV  alpha(A  B   C   E)  margin_E")
for C in [1e-3, 0.01, 0.1, 0.3, 0.5, 0.8, 1.0, 1.5625, 2.0, 2.5, 3.125, 4.0, 10.0, 100.0]:
    m = SVC(C=C, kernel="linear", tol=1e-12).fit(X, y)
    w = m.coef_[0]; w1 = float(w[0])
    sv = m.support_; dc = m.dual_coef_[0]
    alpha = [0.0, 0.0, 0.0, 0.0]
    for j, idx in enumerate(sv):
        alpha[idx] = float(dc[j]) * y[idx]     # alpha_i = dual_coef_j * y_i
    margE = float(y[3] * (X[3] @ w + m.intercept_[0]))
    print("%-8g %+7.4f %+8.4f   %d   (%.4g %.4g %.4g %.4g)   %.3f"
          % (C, w1, float(m.intercept_[0]), len(m.support_),
             alpha[0], alpha[1], alpha[2], alpha[3], margE))

C        w1        b        nSV  alpha(A  B   C   E)  margin_E
0.001    -0.0038  +0.0019   4   (0.001 0.001 0.001 0.001)   -0.003
0.01     -0.0380  +0.0190   4   (0.01 0.01 0.01 0.01)   -0.027
0.1      -0.3800  +0.1900   4   (0.1 0.1 0.1 0.1)   -0.266
0.3      -0.6667  +0.3333   4   (0.2211 0.2211 0.1422 0.3)   -0.467
0.5      -0.6667  +0.3333   4   (0.2944 0.2944 0.08889 0.5)   -0.467
0.8      -0.6667  +0.3333   4   (0.4044 0.4044 0.008889 0.8)   -0.467
1        -0.8000  +0.2000   3   (0.5 0.5 0 1)   -0.360
1.5625   -1.2500  -0.2500   3   (0.7812 0.7812 0 1.562)   -0.000
2        -1.6000  -0.6000   3   (1 1 0 2)   0.280
2.5      -2.0000  -1.0000   3   (1.25 1.25 0 2.5)   0.600
3.125    -2.5000  -1.5000   3   (1.563 1.563 0 3.125)   1.000
4        -2.5000  -1.5000   3   (1.563 1.563 0 3.125)   1.000
10       -2.5000  -1.5000   3   (1.563 1.563 0 3.125)   1.000
100      -2.5000  -1.5000   3   (1.563 1.563 0 3.125)   1.000


In [5]:
# 4구간 구조 검증: ||w||^2 vs C. 두 번 꺾여 수평(4/9, 6.25)이 되는 것이 손계산의 직접적 결과.
Cs = np.array([1e-4, 1e-3, 1e-2, 0.03, 0.05, 0.1, 0.175, 0.3, 0.5, 0.833,
               0.9, 1.0, 1.2, 1.5, 1.5625, 2.0, 2.5, 3.125, 4.0, 6.0, 10.0, 100.0])
w2 = np.array([float(SVC(C=C, kernel="linear", tol=1e-12).fit(X, y).coef_[0] @
                     SVC(C=C, kernel="linear", tol=1e-12).fit(X, y).coef_[0]) for C in Cs])

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.semilogy(Cs, w2, "o-", color="#1d4ed8", lw=2, ms=5, label="실제 \u2016w\u2016\u00b2 (SVC)")
C1 = np.linspace(1e-4, 10/57, 30); ax.plot(C1, 14.44*C1**2, "--", color="#0f5132", lw=1.5, label="I: 14.44 C\u00b2")
C2 = np.linspace(10/57, 5/6, 30);  ax.plot(C2, np.full_like(C2, 4/9), "--", color="#b45309", lw=1.5, label="II: 4/9 (마진 3.0 고정)")
C3 = np.linspace(5/6, 25/8, 30);   ax.plot(C3, 0.64*C3**2, "--", color="#7c3aed", lw=1.5, label="III: 0.64 C\u00b2")
C4 = np.linspace(25/8, 100, 30);   ax.plot(C4, np.full_like(C4, 6.25), "--", color="#dc2626", lw=1.5, label="IV: 6.25 (마진 0.8 고정)")
for Cb, lab in [(10/57, "I\u2192II\n10/57"), (5/6, "II\u2192III\n5/6"), (25/8, "III\u2192IV\n25/8=3.125")]:
    ax.axvline(Cb, color="#737373", ls=":", lw=1.2)
    ax.text(Cb*1.08, 2.0, lab, fontsize=8, ha="left", color="#404040")
ax.set_xlabel("C (위반 허용 vs 마진 최대화의 균형)")
ax.set_ylabel("\u2016w\u2016\u00b2")
ax.set_title("SVM 소프트 마진 4구간 — C를 올려도 마진은 0.8(=2/2.5) 이상 커지지 않는다")
ax.set_ylim(1e-3, 8); ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8, loc="center left")
fig.tight_layout()
fig.savefig(IMG + "/ch08_3_svm_C_effect.svg")
plt.show()

읽어볼 점: 곡선이 **"오르면 수평(4/9) → 다시 오르면 수평(6.25)"으로 두 번
꺾이는** 것이 손으로 구한 4구간표의 직접적 결과다. (1) 구간 I에서
\(\|w\|^2=14.44C^2\)로 오르다 A·B가 마진 위에 닿자(10/57) **4/9로
수평** — 이상치 E의 위반을 \(\alpha_E=C\)로 최대 배정하고 경계가
\(w=-2/3\)로 정지한다. (2) \(C\ge5/6\)에서 C(+2)가 마진에서 빠지고
경계가 \(w=-0.8C\)로 미끄러우며(0.64C²), (3) \(C=25/8\)에서 E도 마진
위에 오면 **6.25로 다시 수평** — 마진 0.8(A와 이상치 E 사이), 경계
x=−0.6. "**C를 키우면 마진이 무한히 커진다**"는 직감은 구간 IV에서
꺾인다 — C가 바꾸는 것은 마진이 아니라 **어떤 점을 SV로 만드는가**
(이상치 E를 마진 안으로 데려올 것인가)다. 확인 문제 7이 바로 이 그림의
질문이다.

## 3. 편향-분산 스펙트럼: 한 데이터, 네 개의 모델 (Ch04, Ch06)

중간고사 빈출 비교 — "같은 데이터에 유연한 모델과 단순한 모델을 놓고
train/val 곡선이 어떻게 달라지는가"를 diabetes(442×10)로 만든다.
Ch06.3 원칙대로: 전처리 `fit`은 train으로만, 모델은 4개,
train/val을 동시에 본다. **val R² 대신 "train R² − val R²" 격차를
주 시선으로** 본다 — 모델이 데이터를 '재는' 정도가 아니라
'외우는' 정도를 재기 때문이다.

In [6]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X, yd = load_diabetes(return_X_y=True)
Xtr, Xtemp, ytr, ytemp = train_test_split(X, yd, test_size=0.4, random_state=0)
Xva, Xte, yva, yte = train_test_split(Xtemp, ytemp, test_size=0.5, random_state=0)
sc = StandardScaler().fit(Xtr)          # <- fit은 train으로만 (Ch06.3)
Xtr, Xva, Xte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

models = {
    "Ridge(\u03bb=1e-4)\n[Ch02,06]": Ridge(alpha=1e-4),
    "Ridge(\u03bb=1)\n[Ch06]": Ridge(alpha=1.0),
    "kNN(k=5)\n[Ch04]": KNeighborsRegressor(n_neighbors=5),
    "Tree(depth=\u221e)\n[Ch07.1]": DecisionTreeRegressor(random_state=0),
}
results = {}
for name, m in models.items():
    m.fit(Xtr, ytr)
    tr, va = r2_score(ytr, m.predict(Xtr)), r2_score(yva, m.predict(Xva))
    results[name.split("\n")[0]] = (tr, va)
    print("%-24s train R\u00b2=%.3f  val R\u00b2=%.3f  gap=%.3f" % (name.replace("\n", " "), tr, va, tr - va))

Ridge(λ=1e-4) [Ch02,06]  train R²=0.579  val R²=0.443  gap=0.137
Ridge(λ=1) [Ch06]        train R²=0.579  val R²=0.442  gap=0.137
kNN(k=5) [Ch04]          train R²=0.618  val R²=0.329  gap=0.290
Tree(depth=∞) [Ch07.1]   train R²=1.000  val R²=-0.014  gap=1.014


In [7]:
names = list(results)
tr = [results[n][0] for n in names]; va = [results[n][1] for n in names]
xpos = np.arange(len(names))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(xpos - 0.2, tr, width=0.38, color="#1d4ed8", label="train R\u00b2")
ax.bar(xpos + 0.2, va, width=0.38, color="#dc2626", label="val R\u00b2")
for i, n in enumerate(names):
    ax.annotate("gap %.2f" % (tr[i] - va[i]), xy=(xpos[i], max(tr[i], va[i]) + 0.06),
                ha="center", fontsize=9, color="#404040")
ax.set_xticks(xpos); ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel("R\u00b2")
ax.set_title("유연성 스펙트럼: Ridge \u2192 kNN \u2192 트리. 오른쪽으로 갈수록 train은 올라가고 val은 벌어진다")
ax.axhline(0, color="#737373", lw=0.8)
ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch08_3_model_spectrum.svg")
plt.show()

읽어볼 점(= 시험 답안 구조):

- **왼쪽 → 오른쪽**으로 갈수록 모델의 유연성(가용 자유도)이
  커진다. Ridge는 10개 가중치에 \(\lambda\)로 매달아 자유도를 조절하는
  "연속 스펙트럼"의 한 점이고, 무제한 트리는 442건을 전부 외우는
  "스펙트럼의 끝"이다.
- gap이 큰 모델(오른쪽)은 **분산**이 크다 — Ch04.1의 언어로는
  "모델 추정량이 train 데이터의 선택에 따라 크게 흔들린다".
  gap이 작은 모델(왼쪽)은 **편향**이 크다 — Ch06의 언어로는
  "평균적으로 진실을 향한 systematic한 오차".
- Tree의 val R²가 **0 아래**(음수)라는 것: train에서 1.00을 찍고도
  검증 데이터에서는 *평균만 예측하는 모델보다 나쁘다*는 뜻이다.
  Ch06.2의 U자 곡선에서 \(\lambda\to 0\) 쪽 끝이 실제로 이렇게
  처져 있음을 보여 주는 것이다.
- 이 그림은 **val 곡선으로 판단하라**는 원칙을 시각화한 것이다 —
  train 곡선만 보면 "Tree가 압승(1.00)"이지만 val 곡선은 정반대다.
  8.1절 노트북 §7의 GBDT 학습 곡선과 같은 메시지다.

## 4. 장을 넘나드는 동치성: GDA와 로지스틱회귀는 정말 같은 경계인가 (Ch02, Ch03)

확인 문제 1의 답은 "둘 다 \(P(y{=}1|x)=\sigma(w^Tx+b)\)에 도달한다"
다. 이 등식이 **실제 데이터**에서 얼마나 타당한지 breast_cancer로
측정해 본다. (1) test AUC가 가까운지, (2) test에서 두 모델의
출력 확률 상관이 높은지, (3) 두 경계 벡터 \(w\)의 방향이 같은
방향인지(코사인 유사도)를 본다. 이론(Ch03.2)은 *정규분포 가정이
성립하면* 계수까지 같은 방향이 나와야 하지만, 실제 데이터는
정규분포가 아니기 때문에 (3)이 크게 어긋나도 (1),(2)가 가깝게
나올 수 있다 — "경계의 형태가 같을 뿐 계수는 경로에 따라 달라진다"
는 답안의 뉘앙스가 이 숫자에 근거한다.

In [8]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score

Xd, yd2 = load_breast_cancer(return_X_y=True)
Xdtr, Xdte, ydtr, ydte = train_test_split(Xd, yd2, test_size=0.3,
                                          random_state=0, stratify=yd2)
sc2 = StandardScaler().fit(Xdtr)
Xtr2, Xte2 = sc2.transform(Xdtr), sc2.transform(Xdte)

lg = LogisticRegression(C=1000.0, max_iter=5000).fit(Xtr2, ydtr)   # 판별적
lda = LinearDiscriminantAnalysis().fit(Xtr2, ydtr)                 # 생성적(GDA)

pl = lg.predict_proba(Xte2)[:, 1]
pd_ = lda.predict_proba(Xte2)[:, 1]
wl, wd = lg.coef_[0], lda.coef_[0]
print("test AUC:  로지스틱=%.4f  GDA=%.4f" %
      (roc_auc_score(ydte, pl), roc_auc_score(ydte, pd_)))
print("test 확률 상관: %.3f" % np.corrcoef(pl, pd_)[0, 1])
print("cos(w_logit, w_GDA) = %.3f  (1.0이면 같은 방향의 경계)" %
      float(wl @ wd / np.linalg.norm(wl) / np.linalg.norm(wd)))

test AUC:  로지스틱=0.9934  GDA=0.9855
test 확률 상관: 0.900
cos(w_logit, w_GDA) = 0.203  (1.0이면 같은 방향의 경계)


읽어볼 점: AUC가 0.99 근처에서 붙어 있고 확률 상관도 높지만
\(w\)의 방향은 어긋난다 — Ch03.2의 동치성은 **"\(P(x|y)\)가
정규분포"라는 전제 하에** 성립하는 등식이다. 전제가 약하게 깨지면
"같은 형태의 경계, 다른 계수"가 된다. 이 구분이 바로 확인 문제 1의
답이 "형태는 같지만 **경로**가 다르다"고 말할 때의 의미다 —
"동치"를 지나치게 강한 말(계수도 같다)로 확장하지 않는 것이
정직한 답안이다.

## 5. Ch06.3의 누수 원칙: 전처리 통계량을 어디에서 fit하는가

8.1절이 요구하는 검증 원칙의 핵심 실수 — `StandardScaler`의
`fit`(평균·표준편차)을 **전체** 데이터로 하는 것 — 의 효과를
diabetes 데이터에서 측정해 본다. 이 데이터가 "행 하나 = 독립
관측"이라서 두 방식이 거의 같게 나온다면, 그것이 오히려
"누수는 언제 커지는가"를 보여주는 예가 된다 — 6.3절의 환자
예처럼 **동일 실체(환자, 계좌 등)가 train과 val/test에
동시에 들어와 정보를 공유할 때** 격차가 벌어지기 때문이다.

In [9]:
sc_all = StandardScaler().fit(np.vstack([Xtr, Xva, Xte]))   # <- 원칙 위반(전체로 fit)
m_all = Ridge(alpha=1.0).fit(sc_all.transform(Xtr), ytr)
m_ok = Ridge(alpha=1.0).fit(Xtr, ytr)                        # Xtr은 이미 train으로 스케일됨

r_leak = r2_score(yva, m_all.predict(sc_all.transform(Xva)))
r_ok = r2_score(yva, m_ok.predict(Xva))
print("val R\u00b2: 스케일러-전체=% .4f  스케일러-train=%.4f  (차이 %.2e)" % (r_leak, r_ok, r_leak - r_ok))
print("-> 이 깨끗한 데이터에서는 차이가 거의 없음. 6.3절의 환자 예처럼")
print("   동일 실체(환자)가 train과 val/test에 모두 들어갈 때 격차가 벌어진다.")

val R²: 스케일러-전체= 0.4417  스케일러-train=0.4417  (차이 3.16e-05)
-> 이 깨끗한 데이터에서는 차이가 거의 없음. 6.3절의 환자 예처럼
   동일 실체(환자)가 train과 val/test에 모두 들어갈 때 격차가 벌어진다.


## 6. 시험 전날 이 노트북을 어떻게 쓰는가

이 노트북의 각 절이 본문의 어떤 시험 유형을 대비하는지 대응표:

| 노트북 절 | 시험 유형 | "손으로 쓸 줄" 확인할 점 |
|---|---|---|
| §1 로지스틱 1스텝 | 유도 문제(빈칸 완성) | \(\hat p=\sigma(\cdot)\)를 먼저 대입하고, \((\hat p-y)\)의 부호를 설명 |
| §2 SVM KKT | "C가 커지면" 정성 질문 | 4구간표: \(C\ge25/8\) 이후 마진이 0.8로 고정되는 이유를 KKT로 설명 |
| §3 스펙트럼 | 모델 비교(정성+숫자) | train/val 격차를 "편향 vs 분산"으로 번역하는 단어 |
| §4 GDA↔logit | 동치성 유도 | 전제(정규분포)를 명시하고, 전제가 깨지면 어떻게 되는지 덧붙임 |
| §5 누수 | 프로젝트/코딩 문제 | `fit`/`transform`을 어디에 쓰는지를 한 문장으로 설명 |

사용법: 각 절의 **코드 셀을 지우고** 먼저 종이에 답을 쓴 뒤, 셀을 다시
실행해서 숫자가 일치하는지 확인한다. **세 숫자**를 종이에 쓸 수 있으면
이 세 장(Ch02, Ch05, Ch04/06/07)의 핵심은 다 잡은 것이다:

1. **§1의 경사값 −0.5507**(Ch02) — \(\frac1n\sum(\hat p-y)x\)의 1/n
   부호와 계수까지.
2. **§2의 \(C^*=25/8=3.125\), 이후 \(\|w\|^2=6.25\) 고정**(Ch05) —
   마진 상한 0.8(=A(−1)과 이상치 E(−0.2) 사이)을 지오메트리로 직접 구하기.
3. **§3의 Tree gap(약 1.0), val R² −0.014**(Ch04/06/07) — train 1.000이
   "좋은 모델"이 아님을 val 곡선으로.

§4·§5는 "이 두 모델/두 방식의 차이는? 전제는? 깨지면?"이라는 문장의
뼈대를 만드는 연습이다.